# Phishing Detection: Fine-Tuning with LoRA
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sjcALane/llm-security-labs/blob/main/notebooks/03_lora_finetuning.ipynb)

Full fine-tuning would update all 494 million weights. LoRA updates about one million and gets most of the benefit.

In [ ]:
# In Colab: Runtime > Change runtime type > T4 GPU. Then run this cell.
try:
    import seclab
except ImportError:
    !pip -q install git+https://github.com/sjcALane/llm-security-labs
    import seclab

# Optional: keep saved adapters between Colab sessions.
# from google.colab import drive
# drive.mount("/content/drive")

from seclab import config, data, lora, metrics, notebook
from seclab.model import load, score
notebook.banner()

In [ ]:
df = data.load_emails()
train, val, test = data.split(df, seed=config.SEED)
test_small = data.sample(test, config.N_TEST, seed=config.SEED)
y = test_small["label"].to_numpy()
emails = test_small["text"].tolist()
print(f"{len(train)} training emails, {len(test_small)} test emails")

## The idea

A linear layer computes $y = Wx$ with $W \in \mathbb{R}^{d_{out}\times d_{in}}$. LoRA freezes $W$ and learns a low-rank correction:

$$y = Wx + \frac{\alpha}{r}\,BAx, \qquad B\in\mathbb{R}^{d_{out}\times r},\quad A\in\mathbb{R}^{r\times d_{in}}$$

The product $BA$ has rank at most $r$, and we use $r = 8$. One matrix gets $r(d_{in}+d_{out})$ trainable parameters instead of $d_{in}d_{out}$. At the start $A$ is random and $B$ is all zeros.

**Question 1.** Why start $B$ at zero? What does the model output before the first training step?

**Your answer:**

## Count the parameters by hand

We put LoRA on four matrices in each attention layer: `q_proj`, `k_proj`, `v_proj` and `o_proj`. Run the next cell to get the sizes you need.

In [ ]:
lm, tok = load(config.MODEL_NAME, config.device())
cfg = lm.config
head_dim = cfg.hidden_size // cfg.num_attention_heads
print("hidden size:", cfg.hidden_size)
print("layers:", cfg.num_hidden_layers)
print("q_proj and o_proj:", cfg.hidden_size, "x", cfg.hidden_size)
print("k_proj and v_proj:", cfg.num_key_value_heads * head_dim, "x", cfg.hidden_size)

`k_proj` and `v_proj` are smaller because this model shares each key and value head across several query heads.

**Question 2.** With $r = 8$ on all four matrices in every layer, how many trainable parameters are there? Work it out, put your number in `my_answer`, and run the check.

In [ ]:
my_answer = 0  # your number here

ft = lora.add_lora(lm, r=8)
trainable, total = lora.count_params(ft)
print(f"trainable: {trainable:,} of {total:,} ({100 * trainable / total:.2f}%)")
print("correct" if my_answer == trainable else "not yet")

## What the model is graded on

Each training example is the prompt followed by the answer, `Yes` or `No`, and an end-of-turn token. Prompt tokens get the label $-100$, which tells PyTorch to leave them out of the loss. The loss is the average negative log-probability of the answer tokens:

$$\mathcal{L} = -\frac{1}{n}\sum_{t\,\in\,\text{answer}} \log p_\theta(x_t \mid x_{<t})$$

In [ ]:
ids, labels = lora.encode_example(tok, "Verify your account at http://example.test", 1)
graded = [tok.decode(i) for i, l in zip(ids, labels) if l != -100]
print(len(ids), "tokens in the example, graded on:", graded)

**Question 3.** Why not train on the prompt tokens too?

**Your answer:**

## Train

This is the slow cell.

In [ ]:
train_small = data.sample(train, config.N_TRAIN, seed=config.SEED)
losses = lora.train(
    ft, tok,
    train_small["text"].tolist(), train_small["label"].tolist(),
    batch_size=config.BATCH, micro_batch=config.MICRO_BATCH, lr=2e-4,
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

k = max(1, len(losses) // 20)
smooth = np.convolve(losses, np.ones(k) / k, mode="valid")
plt.plot(losses, alpha=0.3, label="each step")
plt.plot(range(k - 1, len(losses)), smooth, label=f"average of {k} steps")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

## Evaluate

In [ ]:
p_ft = score(ft, tok, emails)
notebook.save_result("LoRA fine-tuned", metrics.evaluate(y, p_ft))
metrics.table(notebook.load_results())

**Question 4.** Compare the three detectors. Which one would you deploy, and why?

**Your answer:**

Here are some emails the fine-tuned model still gets wrong.

In [ ]:
wrong = test_small.assign(p=p_ft)
wrong = wrong[(wrong["p"] >= 0.5) != (wrong["label"] == 1)]
for _, row in wrong.head(4).iterrows():
    kind = "missed phishing" if row["label"] == 1 else "blocked safe email"
    print(f"--- {kind}, p = {row['p']:.2f}")
    print(row["text"][:400], "\n")

**Question 5.** Pick one of these mistakes. Why do you think the model got it wrong?

**Your answer:**

## Save the adapter

In [ ]:
path = notebook.workdir() / "adapter-clean"
ft.save_pretrained(path)
mb = sum(f.stat().st_size for f in path.iterdir()) / 1e6
print(f"saved to {path}, {mb:.1f} MB")

The base model is about 1 GB. The adapter is a few MB.

**Question 6.** A company wants a separate filter tuned for each of its 500 business customers. What does the adapter size mean for them?

**Your answer:**